In [8]:
# LCEL链式类型的表达
# 其允许输入类型集合对应输出类型集合，使用同样的方法调用组件
# 常见的方法invoke、stream、batch
# 优势：异步支持、内置批量和流式处理支持；可备用设置安全机制；内置日志；自动并行可并行分支

In [9]:
# 设置环境
import os
import openai
from langchain_ollama import ChatOllama

openai.api_key = os.environ.get("DEEPSEEK_API_KEY")
openai.base_url = "https://api.deepseek.com/v1"

In [10]:
# need to pip install pydantic

In [11]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.schema.output_parser import StrOutputParser

In [12]:
prompt = ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)

model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    api_key=os.environ.get("DEEPSEEK_API_KEY") # 需要准备api_key
)

output_parser =StrOutputParser()

In [13]:
chain = prompt | model | output_parser

In [14]:
chain.invoke({"topic": "bears"})

'Why do bears have hairy coats?\n\nBecause they’d look weird in sweaters.'

In [15]:
# 创建更加复杂的链路
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_classic.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OllamaEmbeddings(model="qwen3-embedding:0.6b")
)

retriever = vectorstore.as_retriever()

In [16]:
retriever.invoke("where did harrison work?") # 在检索器上获取相关文档

[Document(metadata={}, page_content='harrison worked at kensho'),
 Document(metadata={}, page_content='bears like to eat honey')]

In [17]:
retriever.invoke("what does bear like to eat")

[Document(metadata={}, page_content='bears like to eat honey'),
 Document(metadata={}, page_content='harrison worked at kensho')]

In [18]:
# 上述操作可以在具有大量文档当中使用，并且返回最相关文档，其将被使用于检索增强生成管道
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [19]:
# 链的唯一输入是用户问题，创建接收单个问题处理
from langchain_classic.schema.runnable import RunnableMap

In [20]:
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
}) | prompt | model | output_parser

In [22]:
chain.invoke({"question":"where did harrison work?"})

'Harrison worked at Kensho.'

In [25]:
inputs = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
})

In [26]:
inputs.invoke({"question":"where did harrison work?"}) # 这两个元素会被传到Prompt当中，进而生成提示值，传递给模型，调用模型返回聊天信息，最后传输给解析器

{'context': [Document(metadata={}, page_content='harrison worked at kensho'),
  Document(metadata={}, page_content='bears like to eat honey')],
 'question': 'where did harrison work?'}

In [27]:
# 添加参数
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [28]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    temperature=0
).bind(functions=functions)

In [29]:
runnable = prompt | model

In [30]:
runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='I don’t have live access to current weather conditions. For the latest forecast in San Francisco, I’d recommend checking a weather service like weather.com, the NWS, or AccuWeather.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 89, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 110, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 89}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'aad32bcc-f849-4506-8a72-e52e6a52775e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07c98-a42b-7e63-a061-81b36630c29b-0', tool_calls=[], inva